# Ingestion bronze

Charge chaque source de `datas` dans une table du schéma `bronze` de PostgreSQL :

- `telemetry.csv` → `bronze.telemetry`
- `releves_incidents.csv` → `bronze.releves_incidents`
- `machine.sql` → `bronze.machine` et `bronze.maintenance` (une table par table du dump)

Chaque table contient :

- les colonnes sources à l'identique, stockées en texte ;
- après chaque colonne, une colonne `<colonne>_type` avec le type inféré ;
- une colonne `hash` : SHA-256 des valeurs sources de la ligne, calculé avant anonymisation ;
- les données de l'opérateur (`operator_name`, `operator_badge`) anonymisées en SHA-256 avec le sel `ANONYMISATION_SALT` du `.env`.

Chaque chargement est tracé dans `bronze.ingestion_log` avec la version du fichier (SHA-256 de ses octets).
Une version déjà chargée est ignorée.

Prérequis : `docker compose up -d db admin`.

In [25]:
import hashlib
import json
import os
import re
from pathlib import Path

import pandas as pd
import sqlglot
from dotenv import load_dotenv
from sqlalchemy import URL, create_engine, text
from sqlglot import exp

load_dotenv(".env")

DATA_DIR = Path("datas")
SCHEMA = "bronze"
LOG_TABLE = "ingestion_log"
ENCODING = "utf-8"
CSV_SEPARATOR = ","
OPERATOR_COLUMNS = {"operator_name", "operator_badge"}
SALT = os.environ.get("ANONYMISATION_SALT", "")
if not SALT:
    raise ValueError("ANONYMISATION_SALT est vide : anonymisation impossible")

## Lecture des sources

Les valeurs sont lues en texte, sans conversion, pour rester identiques à la source.
Le dump SQL n'est pas exécuté : il est analysé avec `sqlglot` et chaque `INSERT` devient une table.
Une valeur `NULL` du dump devient une valeur absente.

In [26]:
def read_csv_source(path: Path) -> dict[str, pd.DataFrame]:
    df = pd.read_csv(path, sep=CSV_SEPARATOR, encoding=ENCODING, dtype=str, keep_default_na=False)
    return {path.stem: df}


def literal_value(value: exp.Expression) -> str | None:
    if isinstance(value, exp.Null):
        return None
    if isinstance(value, exp.Literal):
        return value.this
    return value.sql()


def read_sql_source(path: Path) -> dict[str, pd.DataFrame]:
    tables = {}
    for statement in sqlglot.parse(path.read_text(encoding=ENCODING), read="postgres"):
        if not isinstance(statement, exp.Insert):
            continue
        columns = [column.name for column in statement.this.expressions]
        rows = [[literal_value(v) for v in row.expressions] for row in statement.expression.expressions]
        tables[statement.this.this.name] = pd.DataFrame(rows, columns=columns)
    return tables


def read_sources() -> list[tuple[Path, str, pd.DataFrame]]:
    sources = []
    for path in sorted(DATA_DIR.glob("*.csv")):
        sources += [(path, name, df) for name, df in read_csv_source(path).items()]
    for path in sorted(DATA_DIR.glob("*.sql")):
        sources += [(path, name, df) for name, df in read_sql_source(path).items()]
    return sources

## Inférence des types

Mêmes règles que les cartes d'identité, appliquées à une copie convertie de la colonne
(la colonne stockée reste le texte source).

| Type | Règle |
|---|---|
| IDENTITY | texte au format code : lettres + chiffres (`MACH-01`, `INC-000001`, `OP1002`) |
| DATETIME / DATE / TIME | texte entièrement lisible avec un format de date connu |
| BOOLEAN | entier ne contenant que 0 et 1 |
| INTEGER / FLOAT | valeurs toutes numériques |
| CATEGORY | texte avec 20 valeurs distinctes au plus |
| TEXT | autre texte |

In [27]:
IDENTITY_PATTERN = re.compile(r"^[A-Z]+-?\d+$")
DATE_FORMATS = {
    "DATETIME": ["%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%Y-%m-%d %H:%M", "%Y-%m-%d %H:%M:%S%z"],
    "DATE": ["%Y-%m-%d", "%d/%m/%Y"],
    "TIME": ["%H:%M:%S", "%H:%M"],
}


def parse_dates(values: pd.Series):
    for kind, formats in DATE_FORMATS.items():
        for fmt in formats:
            parsed = pd.to_datetime(values, format=fmt, errors="coerce")
            if parsed.notna().all():
                return kind, parsed
    return None, None


def to_typed(col: pd.Series) -> pd.Series:
    values = col.replace("", None)
    numbers = pd.to_numeric(values, errors="coerce")
    if numbers.notna().sum() != values.notna().sum():
        return values
    if (numbers.dropna() % 1 == 0).all():
        return numbers.astype("Int64")
    return numbers


def infer_type(col: pd.Series) -> str:
    col = to_typed(col)
    values = col.dropna()
    if pd.api.types.is_integer_dtype(col):
        return "BOOLEAN" if set(values.unique()) <= {0, 1} else "INTEGER"
    if pd.api.types.is_float_dtype(col):
        return "FLOAT"
    values = values.astype(str)
    if values.str.match(IDENTITY_PATTERN).all():
        return "IDENTITY"
    kind, _ = parse_dates(values)
    if kind:
        return kind
    return "CATEGORY" if values.nunique() <= 20 else "TEXT"

## Empreinte et anonymisation

- `row_hash` : SHA-256 de la liste des valeurs sources de la ligne, sérialisée en JSON
  (une valeur vide et une valeur absente donnent des empreintes différentes).
- `anonymize` : SHA-256 du sel concaténé à la valeur ; une valeur vide reste vide.
- `file_hash` : SHA-256 des octets du fichier ; il change dès que le fichier change et identifie sa version.

In [28]:
def file_hash(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def row_hash(row: pd.Series) -> str:
    payload = json.dumps(row.tolist(), ensure_ascii=False)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def anonymize(value: str | None) -> str | None:
    if not value:
        return value
    return hashlib.sha256((SALT + value).encode("utf-8")).hexdigest()

## Construction de la table bronze

Ordre des opérations : empreinte sur les valeurs sources, puis anonymisation, puis types
(le type décrit la colonne telle qu'elle est stockée).

In [29]:
def build_bronze(source: pd.DataFrame) -> pd.DataFrame:
    hashes = source.apply(row_hash, axis=1)
    df = source.copy()
    for name in OPERATOR_COLUMNS & set(df.columns):
        df[name] = df[name].map(anonymize)
    bronze = pd.DataFrame(index=df.index)
    for name in df.columns:
        bronze[name] = df[name]
        bronze[f"{name}_type"] = infer_type(df[name])
    bronze["hash"] = hashes
    return bronze

## Écriture dans PostgreSQL et journal des chargements

Connexion via les variables du `.env` (PostgreSQL publié sur `localhost:DB_PORT`).

- `bronze.ingestion_log` : une ligne par table chargée, avec le fichier, sa version (`file_hash`),
  les volumes et la date. Le couple (table, version) est unique : une version déjà chargée est ignorée.
- Une nouvelle version d'un fichier remplace la table bronze correspondante.
- La table et sa ligne de journal sont écrites dans la même transaction : en cas d'erreur, rien n'est écrit.

In [30]:
def get_engine():
    url = URL.create(
        "postgresql+psycopg",
        username=os.environ["DB_USER"],
        password=os.environ["DB_PASSWORD"],
        host="localhost",
        port=int(os.environ["DB_PORT"]),
        database=os.environ["DB_NAME"],
    )
    return create_engine(url)


def create_log_table(engine) -> None:
    with engine.begin() as conn:
        conn.execute(text(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}"))
        conn.execute(text(f"""
            CREATE TABLE IF NOT EXISTS {SCHEMA}.{LOG_TABLE} (
                id SERIAL PRIMARY KEY,
                source_file TEXT NOT NULL,
                table_name TEXT NOT NULL,
                file_hash TEXT NOT NULL,
                rows_read INTEGER NOT NULL,
                rows_inserted INTEGER NOT NULL,
                ingested_at TIMESTAMPTZ NOT NULL DEFAULT now(),
                UNIQUE (table_name, file_hash)
            )"""))


def is_ingested(conn, name: str, digest: str) -> bool:
    query = text(f"SELECT 1 FROM {SCHEMA}.{LOG_TABLE} WHERE table_name = :name AND file_hash = :digest")
    return conn.execute(query, {"name": name, "digest": digest}).first() is not None


def ingest(engine, path: Path, name: str, source: pd.DataFrame) -> str:
    digest = file_hash(path)
    with engine.begin() as conn:
        if is_ingested(conn, name, digest):
            return f"{SCHEMA}.{name} : version {digest[:12]} déjà chargée, ignorée"
        bronze = build_bronze(source)
        bronze.to_sql(name, conn, schema=SCHEMA, if_exists="replace", index=False, method="multi", chunksize=1000)
        conn.execute(
            text(f"""INSERT INTO {SCHEMA}.{LOG_TABLE} (source_file, table_name, file_hash, rows_read, rows_inserted)
                    VALUES (:file, :name, :digest, :read, :inserted)"""),
            {"file": path.name, "name": name, "digest": digest, "read": len(source), "inserted": len(bronze)},
        )
    return f"{SCHEMA}.{name} : version {digest[:12]} chargée, {len(bronze)} lignes"

## Ingestion

In [31]:
engine = get_engine()
create_log_table(engine)

for path, name, source in read_sources():
    print(ingest(engine, path, name, source))

bronze.releves_incidents : version 6c259e47b2e8 déjà chargée, ignorée
bronze.telemetry : version 8049c9a77271 déjà chargée, ignorée
bronze.machine : version c2f4bddf7b7e déjà chargée, ignorée
bronze.maintenance : version c2f4bddf7b7e déjà chargée, ignorée


## Journal des chargements

In [32]:
pd.read_sql(f"SELECT * FROM {SCHEMA}.{LOG_TABLE} ORDER BY id", engine)

,id,source_file,table_name,file_hash,rows_read,rows_inserted,ingested_at
0,1,releves_incidents.csv,releves_incidents,6c259e47b2e8c692413d4b6f3aea8b2b6324c59481eae5...,1245,1245,2026-11-27 08:39:03.769592+00:00
1,2,telemetry.csv,telemetry,8049c9a772717be2865fdedf435e3412e517048203487b...,135626,135626,2026-11-27 08:39:04.827532+00:00
2,3,machine.sql,machine,c2f4bddf7b7ec87a5f261d4fd2bca822e10816c4873770...,15,15,2026-11-27 08:39:41.715127+00:00
3,4,machine.sql,maintenance,c2f4bddf7b7ec87a5f261d4fd2bca822e10816c4873770...,115,115,2026-11-27 08:39:41.778992+00:00


## Contrôle de ce qui a été ingéré

Les cellules suivantes lisent les tables bronze, sans rien modifier.

### Extrait des tables

In [33]:
from IPython.display import Markdown, display

PREVIEW_ROWS = 5
BRONZE_TABLES = ["telemetry", "releves_incidents", "machine", "maintenance"]


def preview(engine, name: str, n: int = PREVIEW_ROWS) -> pd.DataFrame:
    return pd.read_sql(text(f'SELECT * FROM {SCHEMA}."{name}" LIMIT :n'), engine, params={"n": n})


for name in BRONZE_TABLES:
    display(Markdown(f"#### {SCHEMA}.{name}"))
    display(preview(engine, name))

#### bronze.telemetry

,machine_id,machine_id_type,timestamp,timestamp_type,temperature_c,temperature_c_type,pressure_bar,pressure_bar_type,voltage_mean_v,voltage_mean_v_type,rotation_mean_rpm,rotation_mean_rpm_type,pieces_produced,pieces_produced_type,hash
0,MACH-01,IDENTITY,2025-06-01 00:00:00,DATETIME,46.348,FLOAT,198.203,FLOAT,227.568,FLOAT,1541.787,FLOAT,4,INTEGER,c9ba2a113b8b35b9fa3db9de908e576f35977b8e1ae6b2...
1,MACH-01,IDENTITY,2025-06-01 00:00:00,DATETIME,46.332,FLOAT,198.206,FLOAT,227.57,FLOAT,1541.76,FLOAT,4,INTEGER,61ad75ad2c30cc8af5c370b02b54c9f7ac1368a75d6df1...
2,MACH-01,IDENTITY,2025-06-01 01:00:00,DATETIME,48.762,FLOAT,198.295,FLOAT,227.48,FLOAT,1537.86,FLOAT,4,INTEGER,4bfe477c23c30b1bc4e7a656ee2caadb0472f6d2e160f3...
3,MACH-01,IDENTITY,2025-06-01 02:00:00,DATETIME,51.352,FLOAT,199.545,FLOAT,228.68,FLOAT,1584.66,FLOAT,13,INTEGER,ac8b6730d0b7cde9820b8828c79b2bcf2c3f67bd3ef7a2...
4,MACH-01,IDENTITY,2025-06-01 03:00:00,DATETIME,49.512,FLOAT,201.641,FLOAT,228.44,FLOAT,1588.96,FLOAT,10,INTEGER,94e66ffc6863467c77619cbc1c0ce31649440e51b53c38...


#### bronze.releves_incidents

,incident_id,incident_id_type,date,date_type,time,time_type,operator_name,operator_name_type,machine_id,machine_id_type,...,type_surconsommation_type,type_blocage_mecanique,type_blocage_mecanique_type,type_alarme_capteur,type_alarme_capteur_type,type_arret_urgence,type_arret_urgence_type,type_defaut_qualite,type_defaut_qualite_type,hash
0,INC-000001,IDENTITY,2025-06-01,DATE,05:42,TIME,fbb216415765e8748ea9eaea00e998bddab97deef8f996...,CATEGORY,MACH-06,IDENTITY,...,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,6a325d47ead36594cb62b54ec657ee3552d59eed792869...
1,INC-000002,IDENTITY,2025-06-01,DATE,21:08,TIME,f0466dcb7a809965bbe7689f69a1d6595927224a9c4b51...,CATEGORY,MACH-15,IDENTITY,...,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,97a3e24c214dcb9c0b3ad56a8e7ce2d43eb50e9edb10a9...
2,INC-000003,IDENTITY,2025-06-02,DATE,05:43,TIME,fbb216415765e8748ea9eaea00e998bddab97deef8f996...,CATEGORY,MACH-10,IDENTITY,...,BOOLEAN,0,BOOLEAN,1,BOOLEAN,0,BOOLEAN,0,BOOLEAN,891ea2e4340bab98ecd76bc1e129414a6390cd1f038b5c...
3,INC-000004,IDENTITY,2025-06-03,DATE,05:43,TIME,b2c1e9fdcc03b1756db7fc71486e563c5414f9b408b8ea...,CATEGORY,MACH-10,IDENTITY,...,BOOLEAN,0,BOOLEAN,1,BOOLEAN,0,BOOLEAN,0,BOOLEAN,34d13be5b759a1605097a07b4ac303edfc93585d0425c9...
4,INC-000005,IDENTITY,2025-06-04,DATE,01:01,TIME,c4326e2cbb0b03fe2a0d62a69e0fae5b881464ee090649...,CATEGORY,MACH-14,IDENTITY,...,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,0,BOOLEAN,fab2d1f210174e6f6518c3e74f0d0acba504dc080444af...


#### bronze.machine

,machine_code,machine_code_type,commissioning_date,commissioning_date_type,max_daily_capacity,max_daily_capacity_type,max_hourly_capacity_pieces,max_hourly_capacity_pieces_type,model,model_type,production_line,production_line_type,location,location_type,criticality,criticality_type,hash
0,MACH-01,IDENTITY,2021-05-12,DATE,770,INTEGER,48,INTEGER,InduPress-X2,CATEGORY,Ligne-A,CATEGORY,Atelier-2,CATEGORY,MEDIUM,CATEGORY,0dbedb0f1b7c1b58034d00a4b8cfce409d59a787069ac5...
1,MACH-02,IDENTITY,2024-09-07,DATE,800,INTEGER,50,INTEGER,InduPress-X2,CATEGORY,Ligne-A,CATEGORY,Atelier-1,CATEGORY,LOW,CATEGORY,df28cc9cd89cd7d220ca87f7b3bbcde8c43f2c5fa0d0e4...
2,MACH-03,IDENTITY,2019-07-23,DATE,1405,INTEGER,88,INTEGER,InduPress-X1,CATEGORY,Ligne-B,CATEGORY,Atelier-1,CATEGORY,HIGH,CATEGORY,eea3db8e4074349c457b5ff644cef33c7629b0d9b2278e...
3,MACH-04,IDENTITY,2023-01-07,DATE,750,INTEGER,47,INTEGER,InduPress-Z1,CATEGORY,Ligne-C,CATEGORY,Atelier-3,CATEGORY,LOW,CATEGORY,00dffec33acc0c887d4632a18b1e56a7100cb66e4089d5...
4,MACH-05,IDENTITY,2024-03-11,DATE,1380,INTEGER,86,INTEGER,InduPress-X2,CATEGORY,Ligne-C,CATEGORY,Atelier-1,CATEGORY,HIGH,CATEGORY,8480b0eb226b9c85cb87c8f777545b062dd059565bcae5...


#### bronze.maintenance

,maintenance_id,maintenance_id_type,machine_code,machine_code_type,maintenance_at,maintenance_at_type,maintenance_type,maintenance_type_type,action_type,action_type_type,component,component_type,description,description_type,related_incident_id,related_incident_id_type,duration_hours,duration_hours_type,hash
0,1,INTEGER,MACH-01,IDENTITY,2025-06-15 10:00:00+00,DATETIME,proactive,CATEGORY,changement_programme,CATEGORY,capteur température,CATEGORY,Remplacement programmé capteur température,TEXT,None,IDENTITY,2.24,FLOAT,d86fadd62200df10b34148913c3de2e4d4363a8d0aab80...
1,2,INTEGER,MACH-01,IDENTITY,2025-08-14 10:00:00+00,DATETIME,proactive,CATEGORY,changement_programme,CATEGORY,filtre hydraulique,CATEGORY,Remplacement programmé filtre hydraulique,TEXT,None,IDENTITY,2.76,FLOAT,4fa98dbb8dd2b77cee19ea4b87a9df26ed0bf9286ec096...
2,3,INTEGER,MACH-01,IDENTITY,2025-10-13 08:00:00+00,DATETIME,proactive,CATEGORY,changement_programme,CATEGORY,variateur vitesse,CATEGORY,Remplacement programmé variateur vitesse,TEXT,None,IDENTITY,1.74,FLOAT,b69c211c3ee54ca51bee6b32f2e90a7925e882843baf20...
3,4,INTEGER,MACH-01,IDENTITY,2025-12-12 14:00:00+00,DATETIME,proactive,CATEGORY,changement_programme,CATEGORY,capteur pression,CATEGORY,Remplacement programmé capteur pression,TEXT,None,IDENTITY,1.88,FLOAT,49ed01221b6667b4a8ea93ac9fb24c24e044c95bad1ac7...
4,5,INTEGER,MACH-01,IDENTITY,2026-02-10 08:00:00+00,DATETIME,proactive,CATEGORY,changement_programme,CATEGORY,filtre hydraulique,CATEGORY,Remplacement programmé filtre hydraulique,TEXT,None,IDENTITY,1.75,FLOAT,32ac89d123212cbaf455aff3587e2e80319e9d7e9afab7...


### Contrôles de qualité

Chaque contrôle produit une ligne par anomalie, avec la clé de la ligne et le motif en clair.

| Règle | Contrôle |
|---|---|
| Doublon strict | même `hash` qu'une ligne précédente : toutes les valeurs sources sont identiques |
| Doublon de clé | même clé métier qu'une ligne précédente, avec des valeurs qui peuvent différer |
| Valeur vide | valeur vide ou absente dans une colonne obligatoire (`related_incident_id` est facultatif : une maintenance planifiée n'a pas d'incident) |
| Aberrant | mesure (`FLOAT`) hors de `[Q1 - 1,5 × IQR ; Q3 + 1,5 × IQR]`, calculé par machine pour la télémétrie |
| Valeur plafond | mesure égale au minimum ou au maximum de la colonne, répétée : saturation possible du capteur |
| Capacité dépassée | pièces produites dans l'heure supérieures à `max_hourly_capacity_pieces` de la machine |
| Incident d'une autre machine | maintenance rattachée à un incident déclaré sur une autre machine |

Les entiers (identifiants, sévérité de 1 à 5, compteurs) ne passent pas par la détection d'aberrants : la méthode IQR n'a pas de sens sur une échelle ou un identifiant.

**Indice de confiance** : part des lignes sans aucune anomalie.
Une anomalie est un signalement, pas une erreur : rien n'est corrigé ni supprimé.

In [34]:
NATURAL_KEYS = {
    "telemetry": ["machine_id", "timestamp"],
    "releves_incidents": ["incident_id"],
    "machine": ["machine_code"],
    "maintenance": ["maintenance_id"],
}
OPTIONAL_COLUMNS = {"maintenance": ["related_incident_id"]}
OUTLIER_GROUPS = {"telemetry": "machine_id"}
IQR_FACTOR = 1.5
EXAMPLES = 3


def source_columns(df: pd.DataFrame) -> list[str]:
    return [name for name in df.columns if f"{name}_type" in df.columns]


def columns_of_type(df: pd.DataFrame, col_type: str) -> list[str]:
    return [name for name in source_columns(df) if df[f"{name}_type"].iloc[0] == col_type]


def row_keys(name: str, df: pd.DataFrame) -> pd.Series:
    first, *others = NATURAL_KEYS[name]
    keys = df[first].astype(str)
    for column in others:
        keys = keys + " / " + df[column].astype(str)
    return keys


def findings(name: str, df: pd.DataFrame, mask: pd.Series, rule: str, column: str, reasons: pd.Series) -> pd.DataFrame:
    return pd.DataFrame({
        "table": name,
        "ligne": df.index[mask],
        "clé": row_keys(name, df[mask]).to_numpy(),
        "règle": rule,
        "colonne": column,
        "motif": reasons[mask].to_numpy(),
    })

In [35]:
def check_duplicates(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    key = NATURAL_KEYS[name]
    sizes = df.groupby(key)[key[0]].transform("size")
    strict = df["hash"].duplicated()
    by_key = df.duplicated(subset=key)
    return [
        findings(name, df, strict, "Doublon strict", "hash", "hash identique à une ligne précédente : " + df["hash"].str[:12]),
        findings(name, df, by_key, "Doublon de clé", " + ".join(key), "clé présente sur " + sizes.astype(str) + " lignes"),
    ]


def check_missing(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    columns = [c for c in source_columns(df) if c not in OPTIONAL_COLUMNS.get(name, [])]
    return [
        findings(name, df, df[c].isna() | (df[c] == ""), "Valeur vide", c, pd.Series("valeur vide", index=df.index))
        for c in columns
    ]


def check_outliers(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    group = OUTLIER_GROUPS.get(name)
    keys = df[group] if group else pd.Series("toutes lignes", index=df.index)
    results = []
    for c in columns_of_type(df, "FLOAT"):
        values = pd.to_numeric(df[c], errors="coerce")
        q1 = values.groupby(keys).transform(lambda v: v.quantile(0.25))
        q3 = values.groupby(keys).transform(lambda v: v.quantile(0.75))
        iqr = q3 - q1
        low, high = q1 - IQR_FACTOR * iqr, q3 + IQR_FACTOR * iqr
        scope = (group + " " + keys) if group else keys
        reasons = (
            values.astype(str)
            + (values > high).map({True: " > borne haute ", False: " < borne basse "})
            + high.where(values > high, low).round(3).astype(str)
            + " (Q1 " + q1.round(3).astype(str) + ", Q3 " + q3.round(3).astype(str)
            + ", IQR " + iqr.round(3).astype(str) + ", " + scope + ")"
        )
        results.append(findings(name, df, (values < low) | (values > high), "Aberrant", c, reasons))
    return results


def check_capped_values(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    results = []
    for c in columns_of_type(df, "FLOAT"):
        values = pd.to_numeric(df[c], errors="coerce")
        for label, bound in [("minimum", values.min()), ("maximum", values.max())]:
            mask = values == bound
            if mask.sum() > 1:
                reason = f"{bound} = {label} de la colonne, atteint {mask.sum()} fois : saturation possible du capteur"
                results.append(findings(name, df, mask, "Valeur plafond", c, pd.Series(reason, index=df.index)))
    return results


def check_capacity(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    capacity = tables["machine"].set_index("machine_code")["max_hourly_capacity_pieces"].astype(int)
    limit = df["machine_id"].map(capacity)
    pieces = pd.to_numeric(df["pieces_produced"], errors="coerce")
    reasons = pieces.astype(str) + " pièces > capacité horaire " + limit.astype(str) + " de " + df["machine_id"]
    return [findings(name, df, pieces > limit, "Capacité dépassée", "pieces_produced", reasons)]


def check_incident_machine(name: str, df: pd.DataFrame, tables: dict) -> list[pd.DataFrame]:
    incident_machine = tables["releves_incidents"].set_index("incident_id")["machine_id"]
    linked = df["related_incident_id"].map(incident_machine)
    mask = linked.notna() & (linked != df["machine_code"])
    reasons = "incident " + df["related_incident_id"].fillna("") + " déclaré sur " + linked.fillna("") + ", maintenance sur " + df["machine_code"]
    return [findings(name, df, mask, "Incident d'une autre machine", "related_incident_id", reasons)]


GENERIC_CHECKS = [check_duplicates, check_missing, check_outliers, check_capped_values]
TABLE_CHECKS = {"telemetry": [check_capacity], "maintenance": [check_incident_machine]}


def run_checks(tables: dict[str, pd.DataFrame]) -> pd.DataFrame:
    results = []
    for name, df in tables.items():
        for check in GENERIC_CHECKS + TABLE_CHECKS.get(name, []):
            results += check(name, df, tables)
    return pd.concat(results, ignore_index=True)

In [36]:
tables = {name: pd.read_sql_table(name, engine, schema=SCHEMA) for name in BRONZE_TABLES}
anomalies = run_checks(tables)

### Synthèse par table

Nombre de lignes concernées par chaque règle, puis indice de confiance.

In [37]:
def quality_summary(tables: dict[str, pd.DataFrame], anomalies: pd.DataFrame) -> pd.DataFrame:
    by_rule = anomalies.groupby(["table", "règle"])["ligne"].nunique().unstack(fill_value=0)
    flagged = anomalies.groupby("table")["ligne"].nunique()
    rows = pd.Series({name: len(df) for name, df in tables.items()})
    summary = pd.DataFrame({"lignes": rows, "lignes signalées": flagged}).fillna(0).astype(int)
    summary["indice de confiance (%)"] = (100 * (1 - summary["lignes signalées"] / summary["lignes"])).round(2)
    return summary.join(by_rule.reindex(summary.index, fill_value=0))


quality_summary(tables, anomalies)

,lignes,lignes signalées,indice de confiance (%),Aberrant,Capacité dépassée,Doublon de clé,Incident d'une autre machine,Valeur plafond,Valeur vide
machine,15,0,100.00,0,0,0,0,0,0
maintenance,115,24,79.13,13,0,0,24,0,0
releves_incidents,1245,0,100.00,0,0,0,0,0,0
telemetry,135626,43895,67.64,4469,37136,1346,0,157,2828


### Détail par règle et par colonne

Nombre d'anomalies et exemples de motifs.

In [38]:
detail = anomalies.groupby(["table", "règle", "colonne"]).size().rename("anomalies").reset_index()
display(detail)
display(anomalies.groupby(["table", "règle", "colonne"]).head(EXAMPLES)[["table", "clé", "règle", "colonne", "motif"]])

,table,règle,colonne,anomalies
0,maintenance,Aberrant,duration_hours,13
1,maintenance,Incident d'une autre machine,related_incident_id,24
2,telemetry,Aberrant,pressure_bar,1738
3,telemetry,Aberrant,rotation_mean_rpm,1802
4,telemetry,Aberrant,temperature_c,329
5,telemetry,Aberrant,voltage_mean_v,1333
6,telemetry,Capacité dépassée,pieces_produced,37136
7,telemetry,Doublon de clé,machine_id + timestamp,1346
8,telemetry,Valeur plafond,rotation_mean_rpm,120
9,telemetry,Valeur plafond,temperature_c,26


,table,clé,règle,colonne,motif
0,telemetry,MACH-01 / 2025-06-01 00:00:00,Doublon de clé,machine_id + timestamp,clé présente sur 2 lignes
1,telemetry,MACH-01 / 2025-06-03 06:00:00,Doublon de clé,machine_id + timestamp,clé présente sur 2 lignes
2,telemetry,MACH-01 / 2025-06-22 17:00:00,Doublon de clé,machine_id + timestamp,clé présente sur 2 lignes
1346,telemetry,MACH-01 / 2025-06-07 17:00:00,Valeur vide,temperature_c,valeur vide
1347,telemetry,MACH-01 / 2025-06-07 18:00:00,Valeur vide,temperature_c,valeur vide
1348,telemetry,MACH-01 / 2025-06-07 19:00:00,Valeur vide,temperature_c,valeur vide
2240,telemetry,MACH-01 / 2025-06-19 22:00:00,Valeur vide,pressure_bar,valeur vide
2241,telemetry,MACH-01 / 2025-06-19 23:00:00,Valeur vide,pressure_bar,valeur vide
2242,telemetry,MACH-01 / 2025-06-20 00:00:00,Valeur vide,pressure_bar,valeur vide
3235,telemetry,MACH-01 / 2025-06-05 11:00:00,Valeur vide,rotation_mean_rpm,valeur vide


### Extrait des doublons de clé

Lignes partageant la même clé métier, côte à côte : seules les mesures diffèrent, et de peu.

In [39]:
def duplicate_groups(name: str, df: pd.DataFrame, n: int = PREVIEW_ROWS) -> pd.DataFrame:
    key = NATURAL_KEYS[name]
    duplicates = df[df.duplicated(subset=key, keep=False)].sort_values(key)
    return duplicates[source_columns(df) + ["hash"]].head(2 * n)


for name, df in tables.items():
    if df.duplicated(subset=NATURAL_KEYS[name]).any():
        display(Markdown(f"#### {SCHEMA}.{name}"))
        display(duplicate_groups(name, df))

#### bronze.telemetry

,machine_id,timestamp,temperature_c,pressure_bar,voltage_mean_v,rotation_mean_rpm,pieces_produced,hash
0,MACH-01,2025-06-01 00:00:00,46.348,198.203,227.568,1541.787,4,c9ba2a113b8b35b9fa3db9de908e576f35977b8e1ae6b2...
1,MACH-01,2025-06-01 00:00:00,46.332,198.206,227.57,1541.76,4,61ad75ad2c30cc8af5c370b02b54c9f7ac1368a75d6df1...
55,MACH-01,2025-06-03 06:00:00,53.404,201.99,228.31,1671.548,50,26a20e46976b522e4fb6a9e4d34fd91510bbcbd1426e90...
56,MACH-01,2025-06-03 06:00:00,53.422,202.009,228.3,1671.56,50,82cac760e64899195150c7aa3d420dee6e303507422290...
523,MACH-01,2025-06-22 17:00:00,41.062,197.187,226.07,1505.46,11,e36b85e90398c99d81dec972acdcddcd82c7d86bf8b819...
524,MACH-01,2025-06-22 17:00:00,41.055,197.171,226.039,1505.423,11,cbf0ba154412cdc3f9621e48c93d6a408f760d22fe3800...
672,MACH-01,2025-06-28 21:00:00,48.597,201.566,227.564,1598.035,49,2f339f08e040be2f2c5ea5d558df02f5099e9abce067ca...
673,MACH-01,2025-06-28 21:00:00,48.592,201.592,227.57,1598.06,49,e49651d1ea009a7bdea523a37546158087fe42b8cb1eec...
853,MACH-01,2025-07-06 09:00:00,49.842,199.107,226.98,1569.16,14,11ba906c1b3ccffa13a4e9c2b2b2dbdcb965ac86ace4cc...
854,MACH-01,2025-07-06 09:00:00,49.859,199.105,226.943,1569.172,14,90c0dd89b7e028c73f94638d04a2f843bf96269bc0fdd0...
